In [1]:
from pathlib import Path
import sys
import json
import time

import numpy as np
import pandas as pd
import joblib
import onnx
import onnxruntime as ort
import onnxmltools

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Automatically find the project root
PROJECT_DIR = Path.cwd()

while not (PROJECT_DIR / "src").is_dir():
    if PROJECT_DIR.parent == PROJECT_DIR:
        raise FileNotFoundError(
            "Project root not found. Open this notebook "
            "inside the SwiftDrop project."
        )
    PROJECT_DIR = PROJECT_DIR.parent

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data_loader import load_dataset
from src.preprocessing import TARGET, prepare_model_frame

MODELS_DIR = PROJECT_DIR / "models"
RAW_DIR = PROJECT_DIR / "data" / "raw"

PIPELINE_PATH = MODELS_DIR / "xgboost_pipeline.joblib"
DATA_PATH = RAW_DIR / "zomato_cleaned.csv"

print("Project root:", PROJECT_DIR)
print("XGBoost pipeline exists:", PIPELINE_PATH.exists())
print("Raw dataset exists:", DATA_PATH.exists())
print("ONNX Runtime version:", ort.__version__)

Project root: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline
XGBoost pipeline exists: True
Raw dataset exists: True
ONNX Runtime version: 1.31.0


In [2]:
# Load the trained XGBoost pipeline
xgb_pipeline = joblib.load(PIPELINE_PATH)

print("Pipeline type:", type(xgb_pipeline))
print("\nPipeline steps:")

if hasattr(xgb_pipeline, "named_steps"):
    for name, step in xgb_pipeline.named_steps.items():
        print(f"{name}: {type(step)}")
else:
    print("The saved object is not a sklearn Pipeline.")

print("\nPipeline parameters:")
print(xgb_pipeline.get_params().keys())

Pipeline type: <class 'sklearn.pipeline.Pipeline'>

Pipeline steps:
preprocessor: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
model: <class 'xgboost.sklearn.XGBRegressor'>

Pipeline parameters:
dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'preprocessor', 'model', 'preprocessor__n_jobs', 'preprocessor__remainder', 'preprocessor__sparse_threshold', 'preprocessor__transformer_weights', 'preprocessor__transformers', 'preprocessor__verbose', 'preprocessor__verbose_feature_names_out', 'preprocessor__num', 'preprocessor__cat', 'preprocessor__num__memory', 'preprocessor__num__steps', 'preprocessor__num__transform_input', 'preprocessor__num__verbose', 'preprocessor__num__imputer', 'preprocessor__num__imputer__add_indicator', 'preprocessor__num__imputer__copy', 'preprocessor__num__imputer__fill_value', 'preprocessor__num__imputer__keep_empty_features', 'preprocessor__num__imputer__missing_values', 'preprocessor__num__imputer__strategy', 'preprocessor__cat__memo

In [3]:
import numpy as np
import onnxruntime as ort
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the saved validation data and original pipeline
# Use the SAME validation split as Notebook 02

In [5]:
from pathlib import Path

XGB_ONNX_PATH = MODELS_DIR / "xgboost_model.onnx"

print("ONNX file exists:", XGB_ONNX_PATH.exists())
print("ONNX path:", XGB_ONNX_PATH)

ONNX file exists: True
ONNX path: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models\xgboost_model.onnx


In [6]:
# Recreate the exact validation split from Notebook 02
raw = load_dataset(str(DATA_PATH))
frame = prepare_model_frame(raw)

X = frame.drop(columns=[TARGET])
y = frame[TARGET].astype(float)

_, X_valid, _, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Original pipeline predictions
original_preds = xgb_pipeline.predict(X_valid)

# Transform validation data using the fitted preprocessor
preprocessor = xgb_pipeline.named_steps["preprocessor"]
X_valid_transformed = preprocessor.transform(X_valid)

# ONNX Runtime expects float32 input
if hasattr(X_valid_transformed, "toarray"):
    X_valid_transformed = X_valid_transformed.toarray()

X_valid_transformed = np.asarray(
    X_valid_transformed, dtype=np.float32
)

# Load ONNX model and predict
session = ort.InferenceSession(
    str(XGB_ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name
onnx_preds = session.run(
    None,
    {input_name: X_valid_transformed}
)[0].reshape(-1)

# Compare predictions and metrics
max_diff = float(np.max(np.abs(original_preds - onnx_preds)))

print("Validation rows:", len(y_valid))
print("Original XGBoost MAE:",
      mean_absolute_error(y_valid, original_preds))
print("ONNX XGBoost MAE:",
      mean_absolute_error(y_valid, onnx_preds))
print("Original XGBoost RMSE:",
      np.sqrt(mean_squared_error(y_valid, original_preds)))
print("ONNX XGBoost RMSE:",
      np.sqrt(mean_squared_error(y_valid, onnx_preds)))
print("Original XGBoost R²:",
      r2_score(y_valid, original_preds))
print("ONNX XGBoost R²:",
      r2_score(y_valid, onnx_preds))
print("Maximum absolute prediction difference:", max_diff)

assert len(original_preds) == len(onnx_preds)
assert np.allclose(original_preds, onnx_preds, rtol=1e-4, atol=1e-4)

print("\nSUCCESS: ONNX predictions match the original XGBoost model.")

Validation rows: 7793
Original XGBoost MAE: 3.0842244143236988
ONNX XGBoost MAE: 3.0842244932560656
Original XGBoost RMSE: 3.849758362690857
ONNX XGBoost RMSE: 3.8497584634782815
Original XGBoost R²: 0.8258568483239679
ONNX XGBoost R²: 0.8258568392057645
Maximum absolute prediction difference: 9.918212890625e-05

SUCCESS: ONNX predictions match the original XGBoost model.


In [7]:
import time
import numpy as np
import onnxruntime as ort

# Load ONNX model
session = ort.InferenceSession(
    str(XGB_ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

input_name = session.get_inputs()[0].name

# Use the already transformed validation data
X_benchmark = np.asarray(
    X_valid_transformed,
    dtype=np.float32
)

# Warm-up runs (excluded from benchmark)
for _ in range(20):
    session.run(None, {input_name: X_benchmark[:100]})

# Benchmark individual predictions
latencies_ms = []

for i in range(1000):
    sample = X_benchmark[i:i+1]

    start = time.perf_counter()
    session.run(None, {input_name: sample})
    end = time.perf_counter()

    latencies_ms.append((end - start) * 1000)

latencies_ms = np.array(latencies_ms)

print("Benchmark: ONNX Runtime on CPU")
print("Predictions:", len(latencies_ms))
print(f"Mean latency:   {np.mean(latencies_ms):.4f} ms")
print(f"Median latency: {np.median(latencies_ms):.4f} ms")
print(f"P95 latency:    {np.percentile(latencies_ms, 95):.4f} ms")
print(f"P99 latency:    {np.percentile(latencies_ms, 99):.4f} ms")
print(f"Max latency:    {np.max(latencies_ms):.4f} ms")
print(f"Throughput:     {1000 / np.mean(latencies_ms):.2f} predictions/sec")

Benchmark: ONNX Runtime on CPU
Predictions: 1000
Mean latency:   0.0536 ms
Median latency: 0.0349 ms
P95 latency:    0.0944 ms
P99 latency:    0.4194 ms
Max latency:    2.6568 ms
Throughput:     18661.56 predictions/sec
